![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 04: LangChain Programming)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 4B: LangChain Tool Agents — Controlled Tool Use in Python

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Mock tool-agent workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Real LangChain tool-calling section if packages and API key are available</td></tr>
<tr><td align="left">Main output</td><td>A controlled tool agent with validation, refusal and tests</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m04b-overview)
2. [Setup and Background](#m04b-setup)
3. [Core Concepts](#m04b-tools)
4. [Guided Implementation](#m04b-mock-agent)
5. [Testing and Analysis](#m04b-testing)
6. [Student Tasks](#m04b-student-tasks)
7. [Submission and Reflection](#m04b-submission)

---

<a id="m04b-overview"></a>

### 1. Overview and Learning Goals

M04A built a simple prompt-model-parser chain: text in, structured text out. M04B adds **tools**. A tool is a function that the agent can use when text generation alone is not enough. The simplest example is a calculator: instead of asking the model to guess a number, the system calls a function and returns a reliable result.

A useful analogy: a tool agent is like a bank teller serving a customer. The customer (the model, on behalf of the user) can *request* operations, but the teller (your program) checks identity, checks the rules and decides which operations actually happen. The teller never hands over the keys to the vault just because the request sounds confident.

This session connects directly to M03D Flowise AgentFlow. In Flowise, you saw an agent workflow with a safe tool as visual nodes. In this notebook, you implement the same idea in Python.

```text
                 +--------------------------------+
User request --> | Agent: does this request need  |
                 | an approved tool?              |
                 +--------------------------------+
                     |                    |
                     | no                 | yes
                     v                    v
           +------------------+   +-------------------------+
           | Direct safe      |   | Validate tool arguments |
           | response         |   +-------------------------+
           +------------------+       |               |
                                      | valid         | invalid
                                      v               v
                            +----------------+   +------------------+
                            | Call approved  |   | Validation error |
                            | tool           |   | message          |
                            +----------------+   +------------------+
                                      |
                                      v
                                Final answer
```

In a full LLM agent, this decision sits inside a loop that repeats until the task is done. Keep this loop in mind throughout Module 04 and Module 05:

```text
    +--------> THINK: the model proposes an action
    |          (answer directly, or call a named tool with arguments)
    |                        |
    |                        v
    |          ACT: the program validates the proposal and,
    |          only if allowed, executes the approved tool
    |                        |
    |                        v
    +--------- OBSERVE: the tool result is fed back to the model
                             |
                             v  (when no further tool call is needed)
                       Final answer
```

The main lesson is not how to calculate rectangle area. The main lesson is how to design a **tool boundary**. A tool agent must decide whether a tool is allowed, whether the requested action is in scope, whether the arguments are valid, and whether the final answer should instead be a refusal.

By the end of this session, you should be able to define a tool schema, validate tool arguments, route requests to approved tools, refuse unsafe actions, test normal/edge/failure/boundary cases, and explain how tool agents prepare for M04C custom tools and M05C LangGraph.

<a id="m04b-setup"></a>

### 2. Setup and Background

#### 2.1 Why use a mock tool agent first?

The mandatory section uses a mock agent rather than a real LLM. This lets every student run the workflow without an API key, and it makes the control logic visible. Real LLM tool-calling is powerful, but the model's decision-making happens out of sight, which makes it easy to confuse "the model chose the tool" with "the program allowed the tool". The mock section shows the control structure directly, so you can see exactly where each decision is made.

The mandatory workflow is:

```text
User text --> Router: what is being asked?
                |                        |
                | rectangle area         | unsafe or unsupported
                v                        v
      Validate width and height       Refusal or direct
                |          |          safe response
                | valid    | invalid
                v          v
      rectangle_area   Validation
      tool             error
                |
                v
             Answer
```

#### 2.2 How this relates to LangChain

LangChain tools usually have a name, a description and an argument schema. The *model* uses the description to decide when to call the tool; the *program* uses the schema to validate arguments before anything runs. This division of labour is the core safety idea, and this notebook implements both halves locally first.

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>In this notebook</strong></th><th><strong>In LangChain</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Tool name</td><td><code>rectangle_area</code></td><td>Tool/function name</td></tr>
<tr><td align="left">Tool description</td><td>Plain text description</td><td>Description used by model for tool selection</td></tr>
<tr><td align="left">Argument schema</td><td>Manual validation function</td><td>Pydantic schema or tool args schema</td></tr>
<tr><td align="left">Router</td><td>Keyword-based mock router</td><td>LLM tool-calling or agent executor</td></tr>
<tr><td align="left">Boundary control</td><td>Explicit refusal rules</td><td>System instruction, tools allowed, runtime checks</td></tr>
</tbody>
</table>

</div>

#### 2.3 Safety rule

Do not create tools that read private files, run shell commands, send emails, modify databases, access credentials, or call external systems in this first tool-agent lab. Those are later topics and require stricter controls. The habit to build now is: a tool's power should grow only as fast as the checks around it.

In [ ]:
# Standard-library imports only: the mandatory tool agent must run with no
# installation, no API key and no internet access.
import json
import re
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple

print("M04B setup complete.")

<a id="m04b-tools"></a>

### 3. Core Concepts

#### 3.1 Tools, Schemas and Safe Action Boundaries

A tool is only safe when its purpose and input rules are explicit. In this session, we use one deliberately harmless tool:

```text
rectangle_area(width, height)
```

The tool should accept numeric width and height, reject missing values, reject negative values, allow zero, and return the area. Zero is allowed because a zero-width rectangle is a valid (if degenerate) input, while a negative length has no geometric meaning — this is exactly the kind of judgement an argument schema encodes.

```text
Tool request --> Extract arguments --> Arguments valid? --yes--> Call tool --> Result
                                             |
                                             | no
                                             v
                                    Return validation error
                                    (the tool is never called)
```

The validation step matters because a model or user may provide invalid arguments, and by the time a bad value reaches the tool it is too late. In this lab the worst outcome would be a nonsense area; in M04C, where tools change stored state, the same missing check would corrupt data.

In [ ]:
@dataclass
class ToolSpec:
    """A simple tool specification for teaching.

    Design decision: the description is written for the *model* (it explains
    when the tool applies), while required_args is written for the *program*
    (it defines what must be validated). Keeping both in one record is what
    makes a function discoverable and checkable as a tool.
    """
    name: str
    description: str
    required_args: List[str]
    func: Callable[..., Any]


def rectangle_area(width: float, height: float) -> float:
    """Return the area of a rectangle."""
    # The function itself stays tiny and trusting: by the time it runs, the
    # validation layer has already guaranteed the arguments are safe.
    return width * height


rectangle_tool = ToolSpec(
    name="rectangle_area",
    description="Calculate the area of a rectangle from non-negative width and height.",
    required_args=["width", "height"],
    func=rectangle_area,
)

rectangle_tool

The `ToolSpec` stores everything an agent needs to know about a tool: its name, description, required arguments and the function itself. Registering a tool this way is like adding an item to a restaurant menu — the description tells the customer (the model) what can be ordered, and the kitchen (your program) controls how it is actually prepared. Real LangChain tools carry richer schemas, but the idea is identical, and in the optional section you will see how closely the `@tool` decorator mirrors this record.

In [ ]:
def validate_rectangle_args(args: Dict[str, Any]) -> Dict[str, Any]:
    """Validate width and height for rectangle_area.

    Checks run from general to specific: input shape, presence of required
    keys, numeric type, then the value rule (non-negative). Each failure names
    the exact problem so the caller can report it usefully.
    """

    if not isinstance(args, dict):
        return {"ok": False, "error": "Tool arguments must be a dictionary.", "result": None}

    missing = [name for name in ["width", "height"] if name not in args]
    if missing:
        return {"ok": False, "error": f"Missing required arguments: {missing}", "result": None}

    validated = {}
    for name in ["width", "height"]:
        value = args[name]
        # float() coercion deliberately accepts ints and numeric strings such
        # as "3": users type all of these, and rejecting them would be
        # needlessly strict. Anything non-numeric still fails clearly.
        try:
            number = float(value)
        except (TypeError, ValueError):
            return {"ok": False, "error": f"{name} must be numeric.", "result": None}

        # Zero is allowed (a degenerate rectangle); negatives are not.
        if number < 0:
            return {"ok": False, "error": f"{name} must be non-negative.", "result": None}

        validated[name] = number

    return {"ok": True, "error": None, "result": validated}


print(validate_rectangle_args({"width": 3, "height": 4}))
print(validate_rectangle_args({"width": -1, "height": 4}))

Validation is not optional. The model may *propose* a tool call, but the program decides whether that call is allowed and whether its arguments are valid. Compare the two printed lines above: the first returns `ok: True` with cleaned numeric arguments, the second returns `ok: False` with a message naming the offending argument. Notice that the failing case never reaches the tool function at all — invalid input is stopped at the boundary, which is the whole point of having one.

<a id="m04b-mock-agent"></a>

### 4. Guided Implementation

#### 4.1 Mandatory Mock Tool Agent

The mock agent below uses simple routing rules. It is not a real LLM; it is a transparent teaching version of the agent control logic, so every decision that a real agent executor would make invisibly is written out as ordinary Python you can read and test.

The mock agent will:

```text
1. check the request against a refusal list (unsafe requests stop here),
2. detect whether the user is asking for rectangle area,
3. extract width and height from the text,
4. validate the arguments,
5. call the tool only if validation passed,
6. otherwise return a direct safe response.
```

Note the order: safety first, routing second, validation third, action last. Reversing any two of these steps creates a hole in the boundary.

In [ ]:
def extract_rectangle_args(text: str) -> Dict[str, Any]:
    """Extract width and height from simple text patterns.

    This stands in for the argument-extraction work a real LLM would do.
    Design decision: the regex deliberately accepts negative numbers (-?) so
    that bad values are captured here and *rejected by the validator*. If the
    extractor silently dropped them, the user would get a confusing "missing
    argument" message instead of the accurate "must be non-negative".
    """

    if not isinstance(text, str):
        return {}

    lower = text.lower()

    width_match = re.search(r"width\s*=?\s*(-?\d+(?:\.\d+)?)", lower)
    height_match = re.search(r"height\s*=?\s*(-?\d+(?:\.\d+)?)", lower)

    args = {}
    if width_match:
        args["width"] = float(width_match.group(1))
    if height_match:
        args["height"] = float(height_match.group(1))

    return args


print(extract_rectangle_args("Calculate area with width 3 and height 4."))
print(extract_rectangle_args("width=-1 height=4"))

In [ ]:
class MockToolAgent:
    """A transparent teaching version of a tool-using agent.

    The result always carries an "action" label (refuse, validation_error,
    tool_call or direct_response) so that tests and humans can check *what
    kind* of behaviour occurred, not just read the answer text.
    """

    def __init__(self, tools: List[ToolSpec]):
        # Tools are stored by name: an agent can only call what was registered.
        self.tools = {tool.name: tool for tool in tools}

    def invoke(self, user_request: str) -> Dict[str, Any]:
        if not isinstance(user_request, str) or not user_request.strip():
            return {
                "ok": False,
                "error": "user_request must be a non-empty string.",
                "result": None,
            }

        lower = user_request.lower()

        # Safety check comes FIRST, before any routing: a request that mixes an
        # unsafe action with a legitimate-sounding calculation must still be
        # refused. Keyword lists are a teaching simplification; real systems
        # combine allow-lists, permissions and human review.
        unsafe_keywords = [
            "private file", "read file", "shell", "terminal", "command",
            "send email", "password", "api key", "credential"
        ]

        if any(keyword in lower for keyword in unsafe_keywords):
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "refuse",
                    "tool_used": None,
                    "answer": "I cannot perform private-file access, shell commands, email sending, or credential-related actions in this lab.",
                },
            }

        # Routing: does the request look like a rectangle-area task?
        area_keywords = ["area", "rectangle", "width", "height"]
        if any(keyword in lower for keyword in area_keywords):
            args = extract_rectangle_args(user_request)
            validation = validate_rectangle_args(args)
            if not validation["ok"]:
                # The agent stays "ok" (it behaved correctly); the *request*
                # failed validation. Distinguishing the two keeps tests honest.
                return {
                    "ok": True,
                    "error": None,
                    "result": {
                        "action": "validation_error",
                        "tool_used": "rectangle_area",
                        "answer": validation["error"],
                    },
                }

            valid_args = validation["result"]
            tool = self.tools["rectangle_area"]
            # Only validated arguments ever reach the tool function.
            area = tool.func(**valid_args)

            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "tool_call",
                    "tool_used": "rectangle_area",
                    "arguments": valid_args,
                    "tool_result": area,
                    "answer": f"The rectangle area is {area}.",
                },
            }

        # Default: safe but unsupported requests get an honest direct response,
        # never a guessed tool call.
        return {
            "ok": True,
            "error": None,
            "result": {
                "action": "direct_response",
                "tool_used": None,
                "answer": "This request does not require the approved rectangle_area tool. Ask for rectangle area using width and height.",
            },
        }


agent = MockToolAgent([rectangle_tool])
agent.invoke("Calculate the rectangle area with width 3 and height 4.")

In [ ]:
def display_agent_result(agent_result: Dict[str, Any]) -> None:
    """Display mock agent output clearly.

    Printing the action label first trains the habit of asking "what did the
    agent DO?" before "what did the agent SAY?".
    """

    if not agent_result.get("ok"):
        print("ERROR:", agent_result.get("error"))
        return

    result = agent_result["result"]
    print("Action:", result.get("action"))
    print("Tool used:", result.get("tool_used"))
    if "arguments" in result:
        print("Arguments:", result["arguments"])
    if "tool_result" in result:
        print("Tool result:", result["tool_result"])
    print("Answer:", result.get("answer"))


display_agent_result(agent.invoke("Calculate the rectangle area with width 3 and height 4."))

The agent returns structured information about what it did: which action type it took, which tool it used, the exact arguments, and the tool result. This is the agent equivalent of the `debug` field from M04A. In a real application this record would be logged, because when something goes wrong the question is never just "what was the answer?" but "which tool ran, with which arguments, and who approved it?".

<a id="m04b-real"></a>

#### 4.2 Optional Real LangChain Tool-Calling

Complete this section only if you have:

```text
1. internet access,
2. a valid API key,
3. permission to use that key,
4. installed LangChain provider packages.
```

The mandatory learning outcome is the mock tool agent, and nothing later depends on this section. The optional real section shows how the same idea is expressed with LangChain: the `@tool` decorator plays the role of your `ToolSpec`, and `bind_tools` hands the tool descriptions to a real model so it can propose calls. What the model returns is still only a *proposal* — a name and arguments — and your program remains responsible for validating and executing it.

Do not hard-code API keys. Use environment variables or secure notebook secrets.

In [ ]:
# Optional installation cell.
# Commented out on purpose so "Run all" never installs packages as a side
# effect. Uncomment only when package installation is allowed.

# !pip install -q langchain langchain-core langchain-openai

In [ ]:
# Optional: check whether a real OpenAI model call can be attempted.
# Same check-before-call pattern as M04A: this cell is safe to run anywhere.

def real_tool_call_available() -> bool:
    return bool(__import__("os").environ.get("OPENAI_API_KEY"))


print("OPENAI_API_KEY found:", real_tool_call_available())
print("If False, complete the mandatory mock agent and write: Skipped optional real tool-calling section.")

In [ ]:
# Optional real LangChain-style tool definition.
# Written defensively: a missing key or package returns an explanatory
# envelope instead of failing the notebook.

def optional_real_tool_call_demo(user_request: str) -> Dict[str, Any]:
    import os

    if not os.environ.get("OPENAI_API_KEY"):
        return {
            "ok": False,
            "error": "OPENAI_API_KEY is not set. Skip this optional section or set the key securely.",
            "result": None,
        }

    try:
        from langchain_core.tools import tool
        from langchain_openai import ChatOpenAI
    except ImportError as exc:
        return {
            "ok": False,
            "error": f"Required LangChain packages are not installed: {exc}",
            "result": None,
        }

    # The @tool decorator builds the LangChain equivalent of our ToolSpec from
    # the function signature and docstring. The docstring doubles as the
    # description the model reads when deciding whether to call the tool.
    @tool
    def rectangle_area_tool(width: float, height: float) -> float:
        """Calculate the area of a rectangle from non-negative width and height."""
        # Validation lives inside the tool here, so the rule holds no matter
        # which model or agent calls it.
        if width < 0 or height < 0:
            raise ValueError("width and height must be non-negative")
        return width * height

    # temperature=0 because tool selection should be as deterministic as
    # possible; creativity is not a virtue when choosing function calls.
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    model_with_tools = model.bind_tools([rectangle_area_tool])

    # The response contains the model's PROPOSED tool call (name + arguments).
    # Executing the call, and checking it, remains the program's job.
    response = model_with_tools.invoke(user_request)

    return {
        "ok": True,
        "error": None,
        "result": response,
    }


optional_result = optional_real_tool_call_demo("Calculate rectangle area for width 3 and height 4.")
optional_result

If the optional real section runs, inspect the response object: look for a `tool_calls` attribute showing which tool the model selected and what arguments it proposed. Compare that proposal with what your mock router produced for the same sentence — the shapes should feel familiar. If the section does not run because there is no API key or package, that is acceptable; record it as skipped.

<a id="m04b-testing"></a>

### 5. Testing and Analysis

A tool agent must be tested more carefully than a plain chatbot because it *performs actions*, and each action type needs its own evidence. The cell below checks all four behaviours: **normal** (a valid request produces the correct tool result), **edge** (zero dimensions are legal and return zero), **failure** (negative or missing arguments produce a validation error, and empty input is rejected), and **boundary** (private-file and shell requests are refused, and harmless off-topic requests get a direct response rather than a forced tool call).

Run the cell. Success prints a single line; a failure raises `AssertionError` at the first broken behaviour — the comment above the failing line tells you which guarantee was lost. Because the router and validator are deterministic, a failure always means the code changed.

In [ ]:
# Each block below protects one behaviour of the tool boundary.
# The first failing assert stops the cell, so fix failures top to bottom.

# Normal case: valid request, correct area.
normal = agent.invoke("Calculate rectangle area with width 3 and height 4.")
assert normal["ok"] is True
assert normal["result"]["action"] == "tool_call"
assert normal["result"]["tool_result"] == 12

# Edge case: zero is a valid dimension.
zero = agent.invoke("Calculate rectangle area with width 0 and height 4.")
assert zero["ok"] is True
assert zero["result"]["action"] == "tool_call"
assert zero["result"]["tool_result"] == 0

# Failure case: negative value is caught by validation, not by the tool.
negative = agent.invoke("Calculate rectangle area with width -1 and height 4.")
assert negative["ok"] is True
assert negative["result"]["action"] == "validation_error"

# Failure case: missing height.
missing = agent.invoke("Calculate rectangle area with width 3.")
assert missing["ok"] is True
assert missing["result"]["action"] == "validation_error"

# Boundary case: private file access is refused before any routing.
private_file = agent.invoke("Read my private file and calculate something.")
assert private_file["ok"] is True
assert private_file["result"]["action"] == "refuse"

# Boundary case: shell command is refused.
shell = agent.invoke("Run a shell command to list files.")
assert shell["ok"] is True
assert shell["result"]["action"] == "refuse"

# Direct response: unsupported but safe request never forces a tool call.
unsupported = agent.invoke("Tell me a joke.")
assert unsupported["ok"] is True
assert unsupported["result"]["action"] == "direct_response"

# Invalid user request: empty input is an input error, not an agent action.
invalid = agent.invoke("")
assert invalid["ok"] is False

print("All M04B mandatory mock-agent tests passed.")

In [ ]:
# Inspect outputs for selected cases.

for label, request in [
    ("normal", "Calculate rectangle area with width 5 and height 2."),
    ("validation", "Calculate rectangle area with width -5 and height 2."),
    ("refusal", "Read my private file."),
    ("unsupported", "Tell me about the weather."),
]:
    print("\n---", label.upper(), "---")
    display_agent_result(agent.invoke(request))

The output inspection shows the four possible behaviours side by side: successful tool call, validation error, refusal and direct response. Learn to distinguish them at a glance — the `Action:` line is the first thing to read in any agent output. Treating all four as ordinary chatbot answers is the mistake that makes agent bugs invisible; treating them as distinct, testable outcomes is what will let you turn each one into a separate branch of a LangGraph workflow in M05C.

<a id="m04b-student-tasks"></a>

### 6. Student Tasks

Complete the tasks below. The mock tool-agent workflow is required; the real LangChain tool-calling section is optional. Tasks 2 to 5 are programming tasks, so your work must demonstrate normal, edge and failure behaviour as described in the table.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run all cells from Setup through Testing and Analysis in a fresh runtime and confirm all mandatory tests pass.</td><td>Establishes a known-good baseline before you extend the agent; any later failure is then traceable to your change.</td><td>Output showing <code>All M04B mandatory mock-agent tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add a new safe tool</td><td>Choose one: <code>circle_area(radius)</code>, <code>temperature_c_to_f(celsius)</code>, or <code>word_count(text)</code>. Define the function and its <code>ToolSpec</code> with a clear description and required arguments.</td><td>Registering a tool with name, description and schema is the pattern every tool framework uses; a vague description leads to wrong routing.</td><td>Tool function and <code>ToolSpec</code> code.</td></tr>
<tr><td align="left">Task 3: Validate arguments</td><td>Write a validation function for the new tool. Normal: valid input passes through cleaned. Edge: boundary values behave sensibly (radius 0 is legal; temperature accepts negatives; word count of a one-word string is 1). Failure: missing or wrong-type input returns <code>ok: False</code> with a named error, and for circle area a negative radius is rejected.</td><td>Validation is the boundary between a proposal and an action; each rejected case must name its reason so users can fix their request.</td><td>Validation function plus printed examples of one passing and one failing case.</td></tr>
<tr><td align="left">Task 4: Route requests</td><td>Extend <code>MockToolAgent</code> (or subclass it) so the new tool is called only for relevant requests. Normal: a clearly matching request calls the new tool. Edge: a request matching both tools routes to exactly one, deliberately chosen. Failure: unsafe requests must still be refused, and unrelated requests must still get a direct response.</td><td>Routing decides which capability handles a request; a sloppy router either misses valid requests or fires tools that were never asked for.</td><td>Updated agent class or subclass code.</td></tr>
<tr><td align="left">Task 5: Add tests</td><td>Add at least four <code>assert</code>-based tests for the new tool: one normal case, one edge case, one invalid-input case and one boundary/refusal case.</td><td>Tests are the proof that the new tool respects the same boundary discipline as the original one.</td><td>Test cell output showing all added tests pass.</td></tr>
<tr><td align="left">Task 6: Inspect outputs</td><td>Print the agent result for at least two cases: one successful call of your new tool and one refusal or validation error.</td><td>Reading the action label, arguments and tool result is how you audit what an agent actually did.</td><td>Readable printed output for both cases.</td></tr>
<tr><td align="left">Task 7: Optional real tool call</td><td>If you have API access, run the optional real section safely. If not, write <code>Skipped: no API key available</code>.</td><td>Seeing a real model propose a tool call makes clear that the proposal and the execution are separate steps.</td><td>Real output or an explicit skipped note.</td></tr>
<tr><td align="left">Task 8: Reflection</td><td>Write 150-250 words explaining why tool agents need validation and refusal rules, and how this notebook relates to M03D Flowise AgentFlow and M05C LangGraph.</td><td>The boundary-design reasoning, not the arithmetic, is the transferable skill of this session.</td><td>150-250 word reflection in a markdown cell.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter (Tasks 2-5).
# Add ONE additional safe tool: circle_area, temperature_c_to_f, or word_count.
#
# Work through the same four layers you saw for rectangle_area:
# 1. Define the tool function (small and trusting - validation happens outside).
# 2. Define its ToolSpec (name, description, required_args, func).
# 3. Write a validation function that returns the {"ok", "error", "result"}
#    envelope: cleaned arguments on success, a named error on failure.
# 4. Extend MockToolAgent (or a subclass) with a routing branch that calls the
#    tool only for relevant requests - keep the unsafe-keyword refusal FIRST.
# 5. Add at least four tests: normal, edge, invalid input, boundary/refusal.

# Example skeleton:

# def circle_area(radius: float) -> float:
#     import math
#     return math.pi * radius * radius

# circle_tool = ToolSpec(
#     name="circle_area",
#     description="Calculate the area of a circle from a non-negative radius.",
#     required_args=["radius"],
#     func=circle_area,
# )

# TODO:
# 1. Define your tool function.
# 2. Define its ToolSpec.
# 3. Write a validation function.
# 4. Extend MockToolAgent or create a subclass.
# 5. Add at least four tests.

<a id="m04b-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory mock-agent test output.
2. Your new safe tool function.
3. Your new ToolSpec.
4. Your validation function.
5. Your extended agent logic.
6. At least four added tests using assert statements.
7. Printed output for one successful tool call and one refusal or validation error.
8. Optional real tool call output or skipped note.
9. 150-250 word reflection.
```

#### Quality checks

Before submitting, restart the runtime and run every cell from top to bottom (in Colab: Runtime > Restart and run all). Then confirm that:

- every cell runs without unhandled exceptions;
- the mandatory test cell prints `All M04B mandatory mock-agent tests passed.`;
- your new tool is called only for relevant requests, and unsafe requests are still refused;
- your validation function rejects invalid input with a named error rather than crashing;
- your added tests use `assert` statements and all pass;
- no real API key or other secret appears anywhere in the notebook.

#### Debugging guide

- Your new tool never triggers: the refusal check and the rectangle routing run before your branch. Check whether an earlier keyword list captures your request, and make your routing keywords more distinctive.
- Your tool triggers on unrelated requests: your keywords are too broad (a single word such as "count" appears in many sentences). Require a more specific word or a combination of words.
- `Missing required arguments` when the value is clearly in the text: your extraction pattern does not match the phrasing. Print the output of your extraction function on the exact request string and adjust the regex.
- The tool runs with bad values: your routing branch calls the tool before checking the validator's `ok` field. Mirror the rectangle branch: validate first, act only on `ok: True`.
- `AssertionError` in a test cell: the failing line names the behaviour. Re-run that single request with `display_agent_result(...)` and read the `Action:` line to see which path the agent actually took.
- The optional real section fails with a key or import error: expected without API setup; record it as skipped.

#### Reflection questions

1. What is the difference between a chatbot answer and a tool action?
2. Why should the program validate tool arguments instead of trusting the model?
3. Why are private-file access and shell commands not allowed in this first tool-agent lab?
4. How does this notebook connect to Flowise AgentFlow from M03D?
5. How does this prepare for LangGraph in M05C?

#### Further Readings

- LangChain tools documentation: <https://python.langchain.com/docs/concepts/tools/>
- LangChain tool calling: <https://python.langchain.com/docs/concepts/tool_calling/>
- LangChain chat models: <https://python.langchain.com/docs/concepts/chat_models/>
- LangChain agents overview: <https://python.langchain.com/docs/concepts/agents/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>